# ⏳ LangGraph Time-Travel：文字冒險 RPG

用一個迷你 RPG 遊戲，學會 LangGraph 三個殺手級特性：

| 特性 | 在遊戲中的角色 |
|---|---|
| **Checkpointer** | 自動存檔——每一步都是一個存檔點 |
| **`interrupt()` (Human-in-the-loop)** | 暫停遊戲，等玩家做選擇 |
| **Time-Travel** | 「早知道就不打巨龍了」→ 回到過去，改選另一條路，產生平行時間線 |

> 💡 為什麼用遊戲教 time-travel？因為 notebook 一格一格執行的節奏，剛好就是「行動 → 看結果 → 後悔 → 讀檔重來」的遊戲體驗。學員玩過一次就永遠記得 checkpoint fork 的概念。

**環境需求**：本教材以 `langgraph 1.2.7` / `langgraph-checkpoint 4.1.1` 實測通過。完全離線可跑（Mock 故事引擎，不需任何 API key）。

**⚠️ 執行須知**：本 notebook 的 cell **必須由上而下依序執行**——checkpointer 的歷史是累積的，跳著執行或重複執行 resume cell 會產生額外分支，導致後面找 checkpoint 的結果與講義不同。亂掉時最快的復原方式：`Kernel → Restart & Run All`。

In [ ]:
# 只需要 langgraph，本課使用離線 Mock 故事引擎（不需要 API key）
# %pip install -q langgraph grandalf

## 0. 三個必須先分清楚的名詞

Time-travel 新手最容易混淆的就是這三個東西，先建立正確心智模型：

| 名詞 | 遊戲比喻 | 說明 |
|---|---|---|
| `thread_id` | **一個存檔欄位** | 你自己取名（如 `"adventure-1"`）。同一 thread 底下可以長出多條時間線 |
| `checkpoint_id` | **一個存檔點** | 框架自動產生。graph 每執行完一個 super-step 就寫入一個 |
| checkpoint 的內容 | **存檔的實際資料** | 三樣：① 當下完整 state（`values`）② 接下來要執行的 node（`next`）③ 已記錄但尚未反映到 state 的寫入（**pending writes**，稍後會踩到它的坑） |

**Super-step** = graph 一次「節點執行 → 狀態更新」的循環。理解「checkpoint 存在**每個 node 執行完之後**」，是等下正確挑選分岔點的關鍵。

## 1. 定義遊戲狀態（State）

State 就是「存檔內容」：HP、金幣、背包、回合數、劇情記錄。

注意 `log` 用了 `Annotated[list, operator.add]`——這是 **reducer**，每次 node 回傳的 log 會「累加」而不是覆蓋，正好用來記錄冒險史。

In [ ]:
import operator
from typing import TypedDict, Annotated

# GameState 就是「存檔的內容」。graph 每跑完一步，這整包東西就被 checkpointer 存成一個檔。
class GameState(TypedDict):
    hp: int                 # 血量：歸零就 game over
    gold: int               # 金幣
    inventory: list[str]    # 背包（撿到的道具會累加進來）
    turn: int               # 目前第幾回合（1、2、3）
    scene: str              # 本回合的場景敘述（由 narrate node 填入）
    choices: list[str]      # 本回合可選的行動（由 narrate node 填入）
    # log 是唯一「特別」的欄位：Annotated[..., operator.add] 叫做 reducer。
    # 一般欄位 node 回傳新值就「覆蓋」舊值；但有了 operator.add，
    # 每個 node 回傳的 log 會被「串接」到舊 log 後面，而不是蓋掉 → 剛好拿來記錄整段冒險史。
    log: Annotated[list[str], operator.add]
    game_over: bool         # 遊戲是否結束

## 2. Mock 故事引擎

課堂示範用**確定性的劇本**，保證離線可跑、每個學員結果一致。

> 🏋️ 課後作業：把 `narrate` 換成真正的 LLM（見文末延伸練習），劇情就會每次都不一樣。

In [2]:
STORY = {
    1: {
        "scene": "你站在「迷霧森林」入口，遠方傳來狼嚎。地上有一把生鏽的劍。",
        "choices": {
            "撿起劍":   {"inventory": "生鏽的劍", "text": "你撿起劍，感覺踏實多了。"},
            "直接前進": {"hp": -10, "text": "一隻野狼撲出咬傷了你！你徒手趕跑牠。"},
            "原地紮營": {"hp": +5,  "text": "你休息了一晚，恢復精神。"},
        },
    },
    2: {
        "scene": "森林深處出現一座石橋，橋下住著收過路費的巨魔。",
        "choices": {
            "付 20 金幣": {"gold": -20, "text": "巨魔滿意地放行。"},
            "戰鬥":       {"hp": -25, "gold": +30, "text": "一場苦戰！你受了重傷但搜刮了巨魔的錢袋。"},
            "涉水繞過":   {"hp": -5,  "text": "河水冰冷刺骨，你打著哆嗦上了岸。"},
        },
    },
    3: {
        "scene": "你抵達龍之洞窟。巨龍正在沉睡，寶箱就在牠腳邊。",
        "choices": {
            "偷寶箱":   {"gold": +100, "text": "你屏息拿走寶箱——成功了！你發財了！"},
            "攻擊巨龍": {"hp": -40,  "text": "巨龍驚醒，一口龍息把你燒成重傷！"},
            "悄悄離開": {"text": "你決定不冒險，平安走出洞窟。"},
        },
    },
}
MAX_TURN = 3

## 3. 定義 Nodes

兩個 node：
- **`narrate`**：產生本回合的場景與選項
- **`player_turn`**：呼叫 `interrupt()` **暫停整個 graph**，把場景丟給外部（玩家）；玩家的選擇會從 `Command(resume=...)` 傳回來，接著結算效果

`interrupt()` 是 human-in-the-loop 的核心——graph 停在這裡時，狀態已被 checkpointer 完整存檔，你可以關掉程式、明天再玩。

**⚠️ 精確理解 `interrupt()` 的執行語義**（最常被略讀的細節）：resume 之後，`player_turn` 這個 node 會**從頭重新執行一次**，只是這次 `interrupt()` 不再暫停、直接回傳玩家的答案。因此：
- `interrupt()` **之前**的程式碼會執行兩次 → 不要在它前面放有副作用的操作（扣款、寄信、寫 DB）
- 這也是 node 應保持「純函式」風格的原因：所有效果都透過回傳值寫進 state

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command


def narrate(state: GameState):
    """第一個 node：純粹「出題」——根據目前回合數，把場景與選項填進 state。"""
    node = STORY[state["turn"]]                      # 依 turn（1/2/3）取出這一幕的劇本
    # node 只回傳「有改動的欄位」，LangGraph 會自動幫你 merge 回完整 state
    return {"scene": node["scene"], "choices": list(node["choices"].keys())}


def player_turn(state: GameState):
    """第二個 node：暫停等玩家作答，拿到答案後結算 HP／金幣／道具。"""

    # ⏸️ 關鍵一行：interrupt() 會「當場凍結整個 graph」並把括號裡的資料丟給外部（玩家）。
    #    程式執行到這裡就停住、把狀態存檔，直到有人用 Command(resume=選擇) 把它喚醒。
    #    喚醒後，interrupt() 才會「回傳」玩家送進來的那個選擇字串，賦值給 choice。
    #
    # ⚠️ 易踩的坑：resume 之後，player_turn 這個函式會「從第一行重新跑一次」，
    #    只是這次跑到 interrupt() 不再暫停、直接吐出玩家的答案。
    #    → 所以 interrupt() 上面千萬別放有副作用的動作（扣款、寄信），否則會做兩次。
    choice = interrupt({
        "scene": state["scene"],
        "choices": state["choices"],
        "hp": state["hp"],
        "gold": state["gold"],
    })

    # 拿到玩家的選擇後，查劇本得到這個選擇造成的效果（例如 {"hp": -25, "gold": +30, ...}）
    effect = STORY[state["turn"]]["choices"][choice]

    # 把效果套用到目前數值上（effect 裡沒寫的欄位就用 .get(..., 0) 當作 0，即不變）
    hp = state["hp"] + effect.get("hp", 0)
    gold = state["gold"] + effect.get("gold", 0)
    # 若這個選擇會給道具（effect 有 "inventory" 這個 key），就把它 append 進背包
    inv = state["inventory"] + ([effect["inventory"]] if "inventory" in effect else [])

    # 回傳更新後的欄位。turn +1 進入下一幕；log 會被 reducer 累加（不是覆蓋）
    return {
        "hp": hp, "gold": gold, "inventory": inv,
        "turn": state["turn"] + 1,
        "log": [f"[第{state['turn']}回合] 選擇「{choice}」→ {effect['text']} (HP={hp}, 金幣={gold})"],
        "game_over": hp <= 0,                         # 血量歸零 → 結束
    }


def route(state: GameState):
    """條件邊：player_turn 跑完後，決定要「再演一幕」還是「收場」。"""
    if state["game_over"] or state["turn"] > MAX_TURN:
        return END                                    # 死了、或三幕演完 → 結束
    return "narrate"                                  # 🔁 否則回到 narrate，形成迴圈演下一幕

## 4. 組裝 Graph（掛上 Checkpointer = 開啟自動存檔）

`compile(checkpointer=...)` 這一行就是 time-travel 的全部前置條件。之後每個 super-step 都會自動寫入一個 checkpoint。

In [ ]:
builder = StateGraph(GameState)
builder.add_node("narrate", narrate)          # 註冊兩個 node
builder.add_node("player_turn", player_turn)

builder.add_edge(START, "narrate")            # 進場：先 narrate 出題
builder.add_edge("narrate", "player_turn")    # 出完題 → 換玩家作答
builder.add_conditional_edges("player_turn", route)  # 作答完 → 交給 route 決定回 narrate 或 END

# 🔑 掛上 checkpointer 這一行，就是 time-travel 的全部前置條件。
#    有了它，graph 每跑完一個 super-step 都會自動存一個 checkpoint。
#    MemorySaver = 存在記憶體（kernel 重開就清空）；正式環境會換成 SqliteSaver / PostgresSaver。
graph = builder.compile(checkpointer=MemorySaver())

try:
    print(graph.get_graph().draw_ascii())     # 需要 grandalf 套件才畫得出 ASCII 圖
except ImportError:
    print("START → narrate → player_turn ⇄ (route: 回 narrate 或 END)")

## 5. 開始冒險 🗡️

`thread_id` 就是你的「存檔欄位」。第一次 `invoke` 會跑到 `interrupt()` 停下，回傳值裡的 `__interrupt__` 就是給玩家看的場景。

In [ ]:
# thread_id 就是你的「存檔欄位名稱」，自己取。同一個 thread 底下可以長出多條時間線。
cfg = {"configurable": {"thread_id": "adventure-1"}}

def show(result):
    """把 invoke 的回傳值漂亮印出來：可能是『停在玩家選擇』，也可能是『結局』。"""
    # graph 停在 interrupt() 時，回傳值裡會有 "__interrupt__"，裡面是我們丟給玩家看的資料
    if "__interrupt__" in result:
        info = result["__interrupt__"][0].value        # .value 就是 interrupt({...}) 裡那包 dict
        print(f"❤️ HP: {info['hp']}   💰 金幣: {info['gold']}\n")
        print(f"📜 {info['scene']}\n")
        for i, c in enumerate(info["choices"], 1):
            print(f"   {i}. {c}")
    else:
        # 沒有 "__interrupt__" → graph 一路跑到 END，result 就是最終完整 state
        print("🏁 冒險結束！\n")
        for line in result["log"]:
            print("  ", line)
        print(f"\n最終狀態 → HP: {result['hp']}, 金幣: {result['gold']}, 背包: {result['inventory']}")

# 第一次 invoke：傳入初始 state。graph 會跑 narrate → 撞到 player_turn 的 interrupt() 就停下。
# 所以這次回傳的是「第一幕的場景與選項」，不是結局。
result = graph.invoke(
    {"hp": 100, "gold": 50, "inventory": [], "turn": 1,
     "scene": "", "choices": [], "log": [], "game_over": False},
    cfg,
)
show(result)

做出你的選擇——用 `Command(resume=...)` 把答案送回被暫停的 `player_turn`：

In [ ]:
# Command(resume="直接前進") = 把玩家的答案送回剛才暫停的 interrupt()。
# graph 醒來 → 結算第一幕 → 回 narrate 出第二幕 → 又停在 interrupt()。所以又拿到下一幕的選項。
result = graph.invoke(Command(resume="直接前進"), cfg)
show(result)

In [ ]:
# 第二幕作答「戰鬥」→ 結算後進入第三幕（龍之洞窟），再次停在玩家選擇。
result = graph.invoke(Command(resume="戰鬥"), cfg)
show(result)

最後一幕：巨龍與寶箱。我們故意選一個**壞結局**，等一下用 time-travel 反悔。😈

In [ ]:
# 第三幕（最後一幕）作答「攻擊巨龍」→ turn 變成 4 > MAX_TURN，route 回傳 END → 遊戲收場。
# 這次沒有 interrupt，所以 show() 印的是結局。我們故意選這個壞結局，等下用 time-travel 反悔。
result = graph.invoke(Command(resume="攻擊巨龍"), cfg)
show(result)

## 6. 查看時間線：`get_state_history()`

被龍息燒到只剩 25 HP……如果剛才選「偷寶箱」會怎樣？

先看看 checkpointer 幫我們存了哪些檔。**歷史由新到舊排列**，每一列是一個 checkpoint：

In [ ]:
# get_state_history() 回傳這個 thread 的所有 checkpoint（一個 generator），由「新 → 舊」排列。
history = list(graph.get_state_history(cfg))

print(f"共 {len(history)} 個存檔點（新 → 舊）\n")
for s in history:
    # 每個 s 是一個 StateSnapshot，帶三樣重點：
    #   s.values → 當下完整 state（含 turn、hp…）
    #   s.next   → 「接下來要跑哪個 node」，空的代表已經跑到 END
    #   s.config → 這個存檔點的座標（thread_id + checkpoint_id）
    ckpt_id = s.config["configurable"]["checkpoint_id"]
    print(f"turn={s.values.get('turn')}  接下來要執行={s.next or '(結束)'}  id={ckpt_id[:8]}…")

# 💡 為何每個 turn 會出現「兩個」存檔（next=narrate 一個、next=player_turn 一個）？
#    因為 checkpoint 存在「每個 node 執行完之後」。narrate 跑完存一個（下一步是 player_turn），
#    player_turn 跑完又存一個（下一步是 narrate）。這正是等下要挑分岔點的關鍵——
#    我們要挑「next=narrate」那個，也就是玩家還沒作答之前的乾淨存檔點。

### 6.5 解剖一個 checkpoint（知其所以然）

隨便挑一個存檔點，看看裡面到底存了什麼——這對理解等下的分岔操作至關重要：

In [10]:
s = history[2]   # 任挑一個歷史存檔點

print("① config（座標）:")
print("   thread_id     =", s.config["configurable"]["thread_id"])
print("   checkpoint_id =", s.config["configurable"]["checkpoint_id"])
print()
print("② values（當下完整 state）: turn =", s.values["turn"],
      "/ hp =", s.values["hp"], "/ gold =", s.values["gold"])
print()
print("③ next（接下來要執行的 node）:", s.next)
print()
print("④ parent（上一個存檔點，時間線樹靠它串起來）:")
print("  ", s.parent_config["configurable"]["checkpoint_id"])

① config（座標）:
   thread_id     = adventure-1
   checkpoint_id = 1f178751-e5c6-6fbc-8004-aac12289dcb5

② values（當下完整 state）: turn = 3 / hp = 65 / gold = 80

③ next（接下來要執行的 node）: ('narrate',)

④ parent（上一個存檔點，時間線樹靠它串起來）:
   1f178751-6332-6862-8003-487bf1c89321


## 7. ⏳ Time-Travel：回到過去，改變命運

我們要回到「**第 3 回合開始之前**」——也就是 `turn == 3` 且 `next == ('narrate',)` 的那個存檔點。

分岔的正確姿勢是兩步：
1. 用那個 checkpoint 的 config **`invoke(None, ...)`** → 從該點重播，產生一條**新分支**，並再次停在 `interrupt()`
2. 對這條新分支正常 `Command(resume=新選擇)`

> ⚠️ **最重要的坑（本教材實測踩過）**：直覺上你會想「找到當時停在 `interrupt` 的那個存檔點，直接給它一個新的 `Command(resume=新選擇)`」——**這樣行不通**。
>
> **為什麼**：玩家當時的選擇（resume 值）在送出的那一刻，就已經以 **pending write** 的形式記錄在那個 checkpoint 上（回想第 0 節 checkpoint 內容的第 ③ 項）。之後任何從該點出發的重播，框架都會直接套用這個已記錄的舊答案，你新給的 resume 會被忽略——實測結果是它原封不動回傳舊時間線的最終狀態，**連錯誤都不會報**，非常容易誤以為分岔成功了。
>
> **正確心法**：想改變某個決定，就退回到「**那個決定尚未被記錄**」的存檔點——也就是 `interrupt` 所在 node 的**前一步**（本例中 `next == ('narrate',)` 的點）。從那裡重播，才會產生一個乾淨的新 interrupt 讓你重新作答。

In [ ]:
# Step 1: 從歷史裡挑出「第 3 幕開演之前」的那個乾淨存檔點。
#   條件 next == ("narrate",)：narrate 還沒跑、玩家更還沒作答（避開上面 markdown 講的 pending write 坑）
#   條件 turn == 3：鎖定第三幕。next(...) 取符合條件的第一個。
target = next(
    s for s in graph.get_state_history(cfg)
    if s.next == ("narrate",) and s.values.get("turn") == 3
)
print("回到過去：turn =", target.values["turn"], "，此時 HP =", target.values["hp"])

# Step 2: 用「那個存檔點的 config」去 invoke(None, ...)。
#   invoke 的第一個參數傳 None = 不給新輸入，純粹「從這個 checkpoint 重播」。
#   因為 target.config 內含 checkpoint_id，LangGraph 就知道要從這一點岔出一條「新分支」，
#   重跑 narrate → 撞到 player_turn 的 interrupt() 停下 → 得到一個全新、可重新作答的第三幕。
result = graph.invoke(None, target.config)
show(result)

這次我們不打龍了，直接偷寶箱：

In [ ]:
# 這條新分支正常作答。注意 cfg 只有 thread_id、沒有 checkpoint_id——
# 因為上一步 invoke(None, target.config) 已經把「最新位置」移到新分支的 interrupt 上，
# 所以這裡用 cfg 送 resume，答案會落在新分支（偷寶箱），不會回到舊的攻擊巨龍時間線。
result = graph.invoke(Command(resume="偷寶箱"), cfg)
show(result)

In [ ]:
# ✅ 防呆驗證：確認分岔真的成功（眼見為憑）
final = graph.get_state(cfg).values
assert final["gold"] == 180, "分岔失敗！gold=%s（若為 80 代表仍在舊時間線）" % final["gold"]
assert final["hp"] == 65 and "偷寶箱" in final["log"][-1]
print("✅ 驗證通過：目前 thread 最新狀態位於「偷寶箱」時間線（HP 65 / 金幣 180）")

🎉 **同一個存檔（thread），兩條平行時間線**：

- 時間線 A：攻擊巨龍 → HP 25、金幣 80（差點死掉）
- 時間線 B：偷寶箱 → HP 65、金幣 180（發大財）

兩條線的 checkpoint 都還在，隨時可以再回去任何一點開第三條線：

In [ ]:
# 再看一次歷史：存檔點變多了（舊時間線的檔沒被刪，新分支的檔又加進來）。
# checkpoint 只增不減，這就是為什麼兩條時間線能並存。
history = list(graph.get_state_history(cfg))
print(f"分岔後存檔點總數：{len(history)}（原本 8 個 → 兩條時間線並存）\n")

# next == () 代表「已跑到 END」的存檔點，也就是各條時間線的結局。
# 兩條線都收場過，所以這裡會找到兩個結局。
endings = [s for s in history if s.next == ()]
for i, s in enumerate(endings, 1):
    print(f"— 結局 {i}：{s.values['log'][-1]}")

## 8. 課堂討論 & 延伸練習

**這在真實系統裡有什麼用？**
- 🐞 **Debug agent**：agent 在第 7 步做了錯誤決策 → 回到第 6 步，改 state 重跑，不用從頭來
- 🧪 **A/B 測試決策路徑**：同一個對話走兩種策略，比較結果
- 👤 **審核工作流**：主管否決 AI 的建議 → 回到決策點，人工修正後續跑
- 🎮 **產品本身**：互動小說、教學模擬器（例如醫療問診訓練，學生可以「重來一次」）

**延伸練習：**
1. **接上真 LLM**：把 `narrate` 改成呼叫 LLM 生成場景與選項（要求回傳 JSON），`player_turn` 的效果結算也交給 LLM——就是一個 AI Dungeon
2. **持久化存檔**：把 `MemorySaver` 換成 `SqliteSaver`，關掉 notebook 隔天繼續玩（`pip install langgraph-checkpoint-sqlite`）
3. **`update_state()` 作弊碼**：time-travel 不只能改選擇，還能改狀態——回到過去把 `gold` 改成 9999 再重播
4. **畫出時間線樹**：遍歷 `get_state_history()`，用 `parent_config` 把分支結構畫成一棵樹

---

## 📋 附錄：Time-Travel API 速查表

| 操作 | API | 備註 |
|---|---|---|
| 開啟存檔 | `compile(checkpointer=MemorySaver())` | 生產環境換 `SqliteSaver` / `PostgresSaver` |
| 讀最新狀態 | `graph.get_state(config)` | config 只需 `thread_id` |
| 讀歷史 | `graph.get_state_history(config)` | 由新到舊的 generator |
| 回到過去（重播/分岔） | `graph.invoke(None, 該點的.config)` | config 內含 `checkpoint_id` 即代表從該點出發 |
| 改寫歷史 | `graph.update_state(該點的.config, {...})` | 產生帶新值的分岔點（作弊碼） |
| 回覆 interrupt | `graph.invoke(Command(resume=值), config)` | 只對「尚未作答」的 interrupt 有效 |

## 🎓 講師備註

- **時間配置（總長約 50 分鐘）**：概念 10 min → 建圖玩遊戲 15 min → 看歷史+解剖 checkpoint 10 min → 分岔實作與坑點 15 min
- **高頻學員問題**：
  - 「分岔後舊時間線會被刪掉嗎？」→ 不會，checkpoint 只增不減，用 `parent_config` 可追溯完整樹
  - 「`get_state(cfg)` 只給 thread_id 時回傳哪條線？」→ 最新寫入的那條（本例是分岔後的新線，上面的 assert 就是在驗證這件事）
  - 「重複執行 resume cell 會怎樣？」→ 對已結束的 thread 再 invoke 可能產生混亂分支——這就是開頭要求依序執行的原因
- **示範失敗的復原**：`Kernel → Restart & Run All` 一鍵重來（MemorySaver 隨 kernel 重置，教學上反而是優點）